# Taller: Árboles de Decisión con scikit-learn

**Python para Ciencia de Datos — UADE**

### Objetivos de la clase
Al terminar el taller deberíamos poder:

1. Entender **cómo decide** un árbol de clasificación (particiones, impureza, hojas).
2. Ver por qué un árbol sin restricciones **sobreajusta** (overfitting).
3. Controlar la complejidad del árbol con **pre-poda** (`max_depth`) y **post-poda** (`ccp_alpha`).
4. Elegir hiperparámetros con **validación cruzada + Grid Search**, como ya hicimos con Regresión Logística y KNN.
5. Evaluar el modelo con la **curva ROC y el AUC** .

### Agenda
| # | Sección | Tiempo aprox. |
|---|---------|---------------|
| 1 | Librerías, datos y EDA rápido | 10' |
| 2 | Un poco de teoría: ¿cómo crece un árbol? | 15' |
| 3 | El árbol sin poda (y su overfitting) | 10' |
| 4 | Pre-poda: `max_depth` | 15' |
| 5 | Post-poda: `ccp_alpha` | 20' |
| 6 | Modelo de referencia + evaluación | 15' |
| 7 | Otros hiperparámetros (informativo) | 5' |
| 8 | Validación cruzada + Grid Search | 25' |
| 9 | Curva ROC / AUC | 20' |
| 10 | Cierre y ejercicios | 10' |

# 1. Importar librerías

In [ ]:
# En Colab ya viene instalado; descomentar si hace falta
# !pip install scikit-learn seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score, GridSearchCV)

# Árbol de decisión
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# Métricas
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score,
                             roc_curve, roc_auc_score, auc)
from sklearn.preprocessing import label_binarize

RANDOM_STATE = 123

# 2. Cargar el dataset

En Scikit-learn, el dataset Iris presenta las **variables predictoras (X)** y la **variable objetivo (y)** por separado.

- **Variables predictoras (X):** contienen las cuatro características numéricas de las flores: largo y ancho del sépalo, y largo y ancho del pétalo. Son las variables que utiliza el modelo para realizar las predicciones.
- **Variable objetivo (y):** representa la especie de cada flor, es decir, la categoría que el modelo debe aprender a clasificar. Scikit-learn codifica estas categorías mediante etiquetas enteras (*label encoding*): 0 (Setosa), 1 (Versicolor) y 2 (Virginica).

In [ ]:
datos = load_iris()
x = pd.DataFrame(datos.data, columns=datos.feature_names)
y = datos.target
x["species"] = datos.target_names[y]

# Nombres de columnas más cómodos: 'sepal length (cm)' -> 'sepal_length'
x.columns = x.columns.str.replace(" (cm)", "", regex=False).str.replace(" ", "_", regex=False)


## 2.1 EDA

In [ ]:
x.info()

In [ ]:
x.describe()

## Análisis univariado

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

variables = x.drop(columns="species").columns

for ax, variable in zip(axes.flat, variables):
    sns.boxplot(
        data=x,
        x="species",
        y=variable,
        hue="species",
        palette="Set2",
        legend=False,
        ax=ax
    )

    ax.set_title(f"Distribución de {variable} por especie")
    ax.set_xlabel("Especie")
    ax.set_ylabel(variable)

plt.tight_layout()
plt.show()

## Análisis bivariado

In [ ]:
g = sns.pairplot(
    data=x,
    hue="species",
    palette="Set2",
    height=2.5
)

# Ubicar la leyenda en el ángulo inferior derecho
sns.move_legend(
    g,
    "lower right",
    bbox_to_anchor=(1, 0),
    title="Especie"
)

plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(data=x, y="sepal_width", x="petal_length",
                hue=pd.Series(y).map(dict(enumerate(datos.target_names))),
                palette="Set2", s=60)
plt.title("Iris: largo vs ancho de pétalo")
plt.legend(title="Especie")
plt.tight_layout()
plt.show()

> **Pregunta para la clase:** mirando este gráfico, ¿qué reglas del tipo *"si petal_length < ... entonces ..."* propondrían para separar las especies?
> Eso es exactamente lo que va a hacer el árbol: buscar **cortes perpendiculares a los ejes**.
>
>Explorar
>* [Vizlearn App](https://vizlearn.app/ai/classical-ml/decision-tree)

# 3. Un poco de teoría: ¿cómo crece un árbol?

Un árbol de clasificación (algoritmo **CART**, que es el que implementa scikit-learn) divide el espacio de variables en **rectángulos** mediante preguntas binarias del tipo:

$$x_j \le t \;?$$

En cada nodo el algoritmo prueba **todas las variables** y **todos los puntos de corte posibles**, y se queda con el que más "purifica" los nodos hijos.

### Medidas de impureza
Si en un nodo $m$ la proporción de la clase $k$ es $p_{mk}$:

* **Gini** (default en sklearn): $\;G = \sum_k p_{mk}(1-p_{mk})$
* **Entropía**: $\;H = -\sum_k p_{mk}\log_2 p_{mk}$

Ambas valen **0** cuando el nodo es puro (una sola clase) y son máximas cuando las clases están mezcladas en partes iguales.

El corte elegido es el que maximiza la **reducción de impureza** (impureza del padre − promedio ponderado de la impureza de los hijos).

### ¿Cuándo para?
Por defecto, sklearn **sigue dividiendo hasta que todas las hojas son puras** (o no se puede dividir más). Es decir: el árbol puede llegar a "memorizar" el dataset de entrenamiento → **overfitting**.

# 4. El árbol sin poda

In [ ]:
X = x.drop(columns=["species"])  # Variables predictoras
y = x["species"]                 # Variable objetivo

# stratify=y mantiene la proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| Test:", X_test.shape)
pd.Series(y_train).value_counts().sort_index()

In [ ]:
# Creamos el modelo
arbol_full = DecisionTreeClassifier(random_state=RANDOM_STATE)

# Entrenamos
arbol_full.fit(X_train, y_train)

# Imprimimos características del modelo y accuracy
print("Profundidad:", arbol_full.get_depth())
print("Cantidad de hojas:", arbol_full.get_n_leaves())
print("Accuracy train:", accuracy_score(y_train, arbol_full.predict(X_train)))
print("Accuracy test :", accuracy_score(y_test,  arbol_full.predict(X_test)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

# Conjunto de entrenamiento
sns.scatterplot(
    data=X_train,
    x="petal_length",
    y="petal_width",
    hue=y_train,
    hue_order=["setosa", "versicolor", "virginica"],
    palette="Set2",
    s=60,
    ax=axes[0]
)

axes[0].set_title(f"Train ({len(X_train)} observaciones)")
axes[0].set_xlabel("Largo del pétalo (cm)")
axes[0].set_ylabel("Ancho del pétalo (cm)")
axes[0].legend(title="Especie")

# Conjunto de prueba
sns.scatterplot(
    data=X_test,
    x="petal_length",
    y="petal_width",
    hue=y_test,
    hue_order=["setosa", "versicolor", "virginica"],
    palette="Set2",
    s=60,
    ax=axes[1],
    legend=False
)

axes[1].set_title(f"Test ({len(X_test)} observaciones)")
axes[1].set_xlabel("Largo del pétalo (cm)")
axes[1].set_ylabel("Ancho del pétalo (cm)")

plt.suptitle("Iris: distribución de Train y Test", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Graficamos el árbol
plt.figure(figsize=(18, 9))
plot_tree(arbol_full, feature_names=x.columns, class_names=datos.target_names,
          filled=True, rounded=True, impurity=False, fontsize=8)
plt.title("Árbol sin restricciones")
plt.show()

Accuracy de **100% en train** pero bastante menos en test: es la señal típica de **overfitting**. El árbol se adaptó a cada observación de entrenamiento, incluidas las que son "ruido" (las pocas versicolor/virginica que se superponen).

Para evitar el overfitting tenemos que **controlar la complejidad** y para ellos hay que **podar** el árbol, es decir, cortarle ramas.

La pregunta es: ¿hasta dónde podamos? Hay dos estrategias.

# 5. Pre-poda (pre-pruning): `max_depth`

**`max_depth` controla cuánto puede crecer el árbol. Es una técnica de pre-poda (pre-pruning).**

* Se fija **antes** de entrenar: el árbol deja de crecer cuando llega a esa profundidad, aunque las hojas no sean puras.
* Un árbol de profundidad $d$ tiene como máximo $2^d$ hojas.
* **Ventaja:** simple, rápido, fácil de interpretar.
* **Desventaja:** es "miope". Puede frenar una rama cuyo corte actual mejora poco, pero que habilitaría cortes muy buenos más abajo (*efecto horizonte*).

Veamos cómo cambia el desempeño al variar `max_depth`:

# 6. Post-poda (post-pruning): `ccp_alpha`

**`ccp_alpha` controla qué ramas conviene conservar una vez construido el árbol. Es una técnica de post-poda (post-pruning).**

La idea (*Minimal Cost-Complexity Pruning*, Breiman et al., 1984):

1. Se deja crecer el árbol **completo**.
2. Se van **eliminando ramas** que aportan poca mejora en relación a la complejidad que agregan.

Para cada subárbol $T$ se define el costo-complejidad:

$$R_\alpha(T) = R(T) + \alpha\,|T|$$

donde:
* $R(T)$ = impureza total (ponderada) de las hojas → qué tan bien ajusta;
* $|T|$ = cantidad de hojas → qué tan complejo es;
* $\alpha \ge 0$ = **penalización por complejidad**.

Interpretación:
* $\alpha = 0$ → no se penaliza la complejidad → árbol completo (overfitting).
* $\alpha$ grande → cada hoja "cuesta caro" → árbol muy chico (en el extremo, solo la raíz → underfitting).

> `ccp_alpha` es el **equivalente al `cp` de `rpart` en R** (la escala no es idéntica, pero el concepto sí).
> El equivalente a `plotcp()` en Python es `cost_complexity_pruning_path()`.

**Ventaja frente a la pre-poda:** como mira el árbol ya construido, no sufre el *efecto horizonte*: puede conservar una rama que "arranca mal" pero termina bien.

Este gráfico cumple el mismo rol que `plotcp()` en R: buscamos el alfa que da **el menor error** (o, con criterio más conservador, el árbol más simple cuyo error está dentro de 1 desvío estándar del mínimo — la *regla 1-SE*).

> En la sección 9 vamos a hacer esto bien: con **validación cruzada**, sin mirar el test.

# 7. Modelo de referencia: `max_depth=2`, `ccp_alpha=0.01`

Combinamos ambas técnicas: limitamos la profundidad (pre-poda) **y** penalizamos la complejidad (post-poda).

> **Probar en clase:** cambiar `max_depth` a 3 o a `None` y ver cómo cambian el árbol y las métricas.

In [ ]:
# arbol = DecisionTreeClassifier(max_depth=3, ccp_alpha=0.01, random_state=RANDOM_STATE)
arbol = DecisionTreeClassifier(max_depth=6, ccp_alpha=0.01, random_state=RANDOM_STATE)
arbol.fit(X_train, y_train)

y_pred = arbol.predict(X_test)
print("Profundidad:", arbol.get_depth(), "| Hojas:", arbol.get_n_leaves())

# Graficamos el árbol
plt.figure(figsize=(18, 9))
plot_tree(arbol, feature_names=x.columns, class_names=datos.target_names,
          filled=True, rounded=True, impurity=False, fontsize=8)
plt.title("Árbol sin restricciones")
plt.show()

## 7.1 Matriz de confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)


En sklearn las **filas son la clase real** y las **columnas la predicha**.
En R (`caret::confusionMatrix`, `table(pred, real)`) suele ser al revés, así que para comparar con R trasponemos:

In [ ]:
ConfusionMatrixDisplay(cm, display_labels=datos.target_names).plot(cmap="Blues")
plt.title("Matriz de confusión (test)")
plt.show()

## 7.2 Reporte de clasificación

In [ ]:
print(classification_report(y_test, y_pred, target_names=datos.target_names))

Recordemos:
* **Accuracy** = (VP + VN) / (VP + VN + FP + FN) → proporción total de predicciones correctas.
* **Precision** = VP / (VP + FP) → de los que predije positivos, cuántos realmente lo eran.
* **Recall (Sensitivity)** = VP / (VP + FN) → de los que realmente eran positivos, cuántos detecté.
* **F1** = media armónica entre precision y recall.

## 7.3 Visualizar el árbol y sus reglas

In [ ]:
plt.figure(figsize=(12, 6))
plot_tree(arbol, feature_names=x.columns, class_names=datos.target_names,
          impurity=True, filled=True, rounded=True, proportion=False)
plt.tight_layout()
plt.show()

**Cómo leer cada nodo:**
* la **condición** de corte (si se cumple → rama izquierda);
* `gini`: impureza del nodo;
* `samples`: cuántas observaciones de train llegan al nodo;
* `value`: cuántas de cada clase;
* `class`: clase mayoritaria (la que se predice si el nodo fuera hoja).

In [ ]:
print(export_text(arbol, feature_names=list(X.columns)))

## 7.4 Importancia de variables

In [ ]:
imp = pd.Series(arbol.feature_importances_, index=X.columns).sort_values()
imp.plot.barh(figsize=(6, 3), title="Importancia de variables (reducción total de Gini)")
plt.show()

> Con esta configuración el árbol usa **una sola variable** (`petal_length`); las demás tienen importancia 0. ¿Qué pasa con las importancias si subimos `max_depth`?

# 8. Otros hiperparámetros (a modo informativo)

`max_depth` y `ccp_alpha` no son los únicos controles de complejidad. `DecisionTreeClassifier` tiene otros hiperparámetros de **pre-poda** que vale la pena conocer:

| Hiperparámetro | Qué controla | Default |
|---|---|---|
| `min_samples_split` | mínimo de observaciones que debe tener un nodo para **poder dividirse** (equivalente a `minsplit` de `rpart`, que en R vale 20) | 2 |
| `min_samples_leaf` | mínimo de observaciones que debe quedar en **cada hoja** (equivalente a `minbucket` de `rpart`) | 1 |
| `max_leaf_nodes` | cantidad máxima de hojas | `None` |
| `min_impurity_decrease` | solo divide si la reducción de impureza supera este umbral | 0.0 |
| `max_features` | cuántas variables se evalúan en cada corte (clave en Random Forest) | `None` (todas) |
| `criterion` | medida de impureza: `"gini"`, `"entropy"`, `"log_loss"` | `"gini"` |
| `class_weight` | pesos por clase, útil con clases desbalanceadas | `None` |

Hoy no los vamos a ajustar, pero se pueden agregar al Grid Search de la sección siguiente exactamente de la misma manera.

### Documentación y bibliografía
* scikit-learn — **Decision Trees (guía de usuario)**: https://scikit-learn.org/stable/modules/tree.html
* scikit-learn — **API `DecisionTreeClassifier`**: https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
* scikit-learn — **Ejemplo de post-poda con cost complexity pruning**: https://scikit-learn.org/stable/auto_examples/tree/plot_cost_complexity_pruning.html
* James, Witten, Hastie & Tibshirani — *An Introduction to Statistical Learning* (ISLR / ISLP), cap. 8 "Tree-Based Methods". Disponible gratis en https://www.statlearning.com
* Hastie, Tibshirani & Friedman — *The Elements of Statistical Learning*, sección 9.2.
* Breiman, Friedman, Olshen & Stone (1984) — *Classification and Regression Trees* (el libro original de CART).

# 9. Validación cruzada + Grid Search

Hasta ahora elegimos `max_depth` y `ccp_alpha` **mirando el test**. Eso es hacer trampa: el test deja de ser una estimación honesta del desempeño en datos nuevos.

Como ya hicimos con **Regresión Logística** y **KNN**, la forma correcta es:

1. Usar **solo el train**.
2. Dividirlo en *k* folds (**validación cruzada**): entrenar en *k−1*, validar en el restante, rotar.
3. Probar **todas las combinaciones** de hiperparámetros de una grilla (**Grid Search**) y quedarnos con la de mejor promedio en validación.
4. Recién al final, evaluar **una sola vez** en test.

Usamos `StratifiedKFold` para que cada fold mantenga la proporción de clases.

## 9.1 Validación cruzada de un modelo

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(
    DecisionTreeClassifier(max_depth=2, ccp_alpha=0.01, random_state=RANDOM_STATE),
    X_train, y_train, cv=cv, scoring="accuracy")

print("Accuracy por fold:", scores.round(3))
print(f"Promedio: {scores.mean():.3f} ± {scores.std():.3f}")

## 9.2 Grid Search sobre `max_depth` y `ccp_alpha`

In [ ]:
param_grid = {
    "max_depth": [1, 2, 3, 4, 5, None],
    "ccp_alpha": [0.0, 0.005, 0.01, 0.02, 0.05, 0.1],
    # "min_samples_leaf": [1, 5, 10],   # <- se podría agregar (sección 8)
    # "criterion": ["gini", "entropy"],
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True,
)
grid.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid.best_params_)
print(f"Mejor accuracy en CV: {grid.best_score_:.3f}")

In [ ]:
res = pd.DataFrame(grid.cv_results_)
cols = ["param_max_depth", "param_ccp_alpha", "mean_train_score",
        "mean_test_score", "std_test_score", "rank_test_score"]
res[cols].sort_values("rank_test_score").head(10).round(3)

> En `cv_results_`, `mean_test_score` es el score en los **folds de validación** (no en nuestro test).

In [ ]:
tabla = res.pivot_table(index=res["param_max_depth"].astype(str),
                        columns="param_ccp_alpha", values="mean_test_score")
plt.figure(figsize=(8, 4.5))
sns.heatmap(tabla, annot=True, fmt=".3f", cmap="viridis")
plt.title("Accuracy promedio en CV (5 folds)")
plt.xlabel("ccp_alpha"); plt.ylabel("max_depth")
plt.show()

> **Para discutir:**
> * ¿Hay muchas combinaciones empatadas? En ese caso, ¿cuál elegirían? (Criterio de parsimonia: **el árbol más simple**.)
> * ¿Qué pasa con `max_depth=None` cuando `ccp_alpha` crece? → la post-poda "compensa" la falta de pre-poda.

## 9.3 Evaluación final en test

In [ ]:
mejor_arbol = grid.best_estimator_   # GridSearchCV ya lo reentrenó con todo el train (refit=True)

y_pred_best = mejor_arbol.predict(X_test)
print("Profundidad:", mejor_arbol.get_depth(), "| Hojas:", mejor_arbol.get_n_leaves())
print(f"Accuracy en test: {accuracy_score(y_test, y_pred_best):.3f}\n")
print(classification_report(y_test, y_pred_best, target_names=datos.target_names))

### ¿Y por qué en test da peor que el árbol de profundidad 2?
Puede pasar, y es una buena oportunidad para discutir:
* El test tiene **30 observaciones**: cada error mueve el accuracy **3.3 puntos**. La diferencia entre 0.83 y 0.90 son **2 flores**.
* La estimación por CV (promedio de 5 folds sobre 120 obs.) es **más estable** que un único test chico.
* Además, en el ranking de la tabla de arriba hay **muchos empates** en el primer puesto. `GridSearchCV` se queda con el primero que encuentra, que no necesariamente es el más simple.

## 9.4 Desempates: criterio de parsimonia (regla 1-SE)
La regla de **1 error estándar** (la misma que se usa con `plotcp()` en R) dice: entre todos los modelos cuyo score está a menos de un desvío del mejor, **elegir el más simple**. Medimos "simple" por cantidad de hojas.

In [ ]:
mejor = res.loc[res["rank_test_score"] == 1].iloc[0]
umbral = mejor["mean_test_score"] - mejor["std_test_score"]

candidatos = res[res["mean_test_score"] >= umbral].copy()
candidatos["hojas"] = [
    DecisionTreeClassifier(random_state=RANDOM_STATE, **p).fit(X_train, y_train).get_n_leaves()
    for p in candidatos["params"]
]
candidatos = candidatos.sort_values(["hojas", "mean_test_score"], ascending=[True, False])
print(f"Umbral 1-SE: {umbral:.3f}")
candidatos[["param_max_depth", "param_ccp_alpha", "hojas", "mean_test_score"]].head(8).round(3)

In [ ]:
params_1se = candidatos.iloc[0]["params"]
arbol_1se = DecisionTreeClassifier(random_state=RANDOM_STATE, **params_1se).fit(X_train, y_train)

print("Hiperparámetros (1-SE):", params_1se)
print("Hojas:", arbol_1se.get_n_leaves())
print(f"Accuracy en test: {accuracy_score(y_test, arbol_1se.predict(X_test)):.3f}")

> El árbol más simple suele **generalizar igual o mejor** y es mucho más fácil de explicar. Esta es la lógica de la post-poda llevada a la selección de modelos.

### Bonus: alfas "candidatos" desde el camino de poda
En lugar de inventar valores de `ccp_alpha`, podemos usar los **alfas efectivos** que calculamos en la sección 6.1 como grilla:

In [ ]:
grid_alpha = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid={"ccp_alpha": ccp_alphas[:-1]},
    cv=cv, scoring="accuracy", n_jobs=-1)
grid_alpha.fit(X_train, y_train)

r = pd.DataFrame(grid_alpha.cv_results_)
plt.figure(figsize=(7, 4))
plt.errorbar(r["param_ccp_alpha"].astype(float), r["mean_test_score"],
             yerr=r["std_test_score"], fmt="o-", capsize=3)
plt.xlabel("ccp_alpha"); plt.ylabel("Accuracy CV")
plt.title("Equivalente a plotcp() de R, con validación cruzada")
plt.grid(alpha=.3); plt.show()

print("Mejor ccp_alpha:", grid_alpha.best_params_)

# 10. Curva ROC y AUC

### Repaso
La **curva ROC** grafica, para **todos los umbrales** de decisión posibles:
* eje X: **FPR** (tasa de falsos positivos) = FP / (FP + VN) = 1 − especificidad
* eje Y: **TPR** (tasa de verdaderos positivos) = VP / (VP + FN) = recall / sensibilidad

El **AUC** (área bajo la curva) resume la curva en un número:
* AUC = 0.5 → el modelo no es mejor que tirar una moneda;
* AUC = 1 → separa perfectamente las clases.
* Interpretación probabilística: es la probabilidad de que, tomando un positivo y un negativo al azar, el modelo le asigne mayor score al positivo.

### Dos detalles importantes para árboles

**1. Necesitamos probabilidades, no clases.** Usamos `predict_proba`. En un árbol, la probabilidad de cada clase es **la proporción de esa clase en la hoja** a la que cae la observación.
Consecuencia: un árbol con pocas hojas produce **pocos valores distintos de probabilidad** → la curva ROC tiene **pocos escalones** (no es suave como la de una regresión logística).

**2. Iris tiene 3 clases.** La ROC está definida para problemas binarios, así que usamos la estrategia **One-vs-Rest (OvR)**: una curva por clase ("setosa vs. resto", "versicolor vs. resto", "virginica vs. resto") y después promediamos los AUC (*macro-average*).

In [ ]:
y_score = mejor_arbol.predict_proba(X_test)   # matriz n_test x 3
pd.DataFrame(y_score, columns=datos.target_names).round(3).head(10)

In [ ]:
print("Valores distintos de probabilidad que produce el árbol:")
np.unique(y_score.round(3), axis=0)

## 10.1 Curvas ROC One-vs-Rest

In [ ]:
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])   # one-hot de la clase real

plt.figure(figsize=(6.5, 6))
for k, nombre in enumerate(datos.target_names):
    fpr, tpr, _ = roc_curve(y_test_bin[:, k], y_score[:, k])
    plt.plot(fpr, tpr, lw=2, label=f"{nombre} vs resto (AUC = {auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1, label="Azar (AUC = 0.5)")
plt.xlabel("FPR (1 - especificidad)")
plt.ylabel("TPR (sensibilidad)")
plt.title("Curva ROC One-vs-Rest — Árbol de decisión (test)")
plt.legend(loc="lower right"); plt.grid(alpha=.3)
plt.show()

In [ ]:
auc_macro = roc_auc_score(y_test, y_score, multi_class="ovr", average="macro")
auc_weighted = roc_auc_score(y_test, y_score, multi_class="ovr", average="weighted")
print(f"AUC OvR macro   : {auc_macro:.3f}")
print(f"AUC OvR weighted: {auc_weighted:.3f}")

> **Para discutir:**
> * ¿Por qué *setosa* tiene AUC = 1? (Mirar el scatterplot de la sección 2.)
> * ¿Por qué las curvas tienen tan pocos escalones?
> * La dificultad está en *versicolor* vs *virginica*, que se superponen: ahí es donde el AUC cae.
> * Con 30 observaciones de test, cada error mueve mucho la métrica: tomar estos números con cautela.

## 10.2 AUC como métrica del Grid Search

Podemos usar el AUC directamente como criterio para elegir hiperparámetros con `scoring="roc_auc_ovr"`.
A diferencia del accuracy, el AUC evalúa la **calidad de las probabilidades** (el ranking), no solo la clase final.

In [ ]:
grid_auc = GridSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid, cv=cv,
    scoring="roc_auc_ovr", n_jobs=-1)
grid_auc.fit(X_train, y_train)

print("Mejores hiperparámetros (AUC):", grid_auc.best_params_)
print(f"Mejor AUC en CV: {grid_auc.best_score_:.3f}")
print(f"AUC en test    : {roc_auc_score(y_test, grid_auc.predict_proba(X_test), multi_class='ovr'):.3f}")

> **Pregunta:** ¿eligió los mismos hiperparámetros que con accuracy? ¿Por qué podría elegir un árbol más profundo?
> (Pista: más hojas → más valores distintos de probabilidad → mejor ranking.)

# 11. Cierre

### Resumen
* Un árbol divide el espacio con cortes del tipo $x_j \le t$, eligiendo el que más reduce la impureza (Gini / entropía).
* Sin restricciones, **sobreajusta**: hojas puras, 100% en train.
* **Pre-poda** (`max_depth`, `min_samples_split`, `min_samples_leaf`, …): frena el crecimiento antes de entrenar. Simple, pero miope.
* **Post-poda** (`ccp_alpha`): crece el árbol completo y elimina las ramas que no justifican su complejidad. Equivalente al `cp` de R.
* Los hiperparámetros se eligen con **validación cruzada + Grid Search**, nunca mirando el test.
* Para la **ROC/AUC** usamos `predict_proba`; en árboles las probabilidades son proporciones en las hojas (curvas escalonadas). En multiclase: **One-vs-Rest**.

### Ventajas y desventajas de los árboles
| ✅ Ventajas | ❌ Desventajas |
|---|---|
| Muy interpretables (reglas si/entonces) | Alta varianza: pequeños cambios en los datos → árbol distinto |
| No necesitan escalar variables | Fronteras de decisión "en escalera" |
| Capturan interacciones y no linealidades | Tienden a sobreajustar si no se podan |
| Dan importancia de variables | Probabilidades poco "finas" |

La alta varianza es justamente lo que motiva los **ensambles** (Random Forest, Gradient Boosting), que combinan muchos árboles.

# 12. Ejercicios propuestos

1. **Comparar criterios:** agregar `"criterion": ["gini", "entropy"]` a la grilla. ¿Cambia el mejor modelo?
2. **Otros hiperparámetros:** agregar `min_samples_leaf: [1, 5, 10]` y `min_samples_split: [2, 10, 20]`. ¿Qué combinación gana? ¿Es más simple el árbol resultante?
3. **Comparar con modelos anteriores:** entrenar con el mismo `cv` una Regresión Logística y un KNN (con su propio Grid Search) y comparar accuracy y AUC en test. ¿Cuál elegirían y por qué?
4. **Un dataset más desafiante:** repetir el flujo (poda + Grid Search + ROC) con `sklearn.datasets.load_breast_cancer()` (problema **binario**). Acá la ROC es una sola curva y las diferencias entre modelos se notan más.
5. **Estabilidad:** cambiar el `random_state` del `train_test_split` varias veces y observar cómo cambia el árbol. ¿Qué nos dice esto sobre la varianza del modelo?